# SQLite Database

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

PROCESSED_DIR = Path("..") / "data" / "processed"
DB_PATH = Path("..") / "database" / "kenya_maize.db"
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

county_long = pd.read_csv(PROCESSED_DIR / "county_maize_long.csv")
national_afa = pd.read_csv(PROCESSED_DIR / "national_afa_clean.csv")

## 1. Create the tables

In [2]:
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")

conn.executescript("""
DROP TABLE IF EXISTS county_production;
DROP TABLE IF EXISTS counties;
DROP TABLE IF EXISTS national_production;

CREATE TABLE counties (
    county_id       INTEGER PRIMARY KEY,
    county_name     TEXT    NOT NULL UNIQUE,
    small_area_flag INTEGER NOT NULL
);

CREATE TABLE county_production (
    county_id    INTEGER NOT NULL REFERENCES counties(county_id),
    year         INTEGER NOT NULL,
    area_ha      REAL    NOT NULL CHECK (area_ha > 0),
    production_t REAL    NOT NULL CHECK (production_t >= 0),
    PRIMARY KEY (county_id, year)
);

CREATE TABLE national_production (
    year                   INTEGER PRIMARY KEY,
    area_ha                REAL,
    production_t           REAL,
    yield_bags_ha          REAL,
    farmgate_price_ksh_bag REAL,
    imports_t              REAL,
    source                 TEXT NOT NULL
);
""")

## 2. Insert the data

In [3]:
counties = county_long[["county", "small_area_flag"]].drop_duplicates().sort_values("county")
conn.executemany(
    "INSERT INTO counties (county_name, small_area_flag) VALUES (?, ?)",
    [(name, int(flag)) for name, flag in counties.itertuples(index=False)],
)

county_ids = dict(conn.execute("SELECT county_name, county_id FROM counties").fetchall())
conn.executemany(
    "INSERT INTO county_production (county_id, year, area_ha, production_t) VALUES (?, ?, ?, ?)",
    [
        (county_ids[row.county], int(row.year), float(row.area_ha), float(row.production_t))
        for row in county_long.itertuples(index=False)
    ],
)

conn.executemany(
    "INSERT INTO national_production VALUES (?, ?, ?, ?, ?, ?, ?)",
    [
        (int(r.year), r.area_ha, r.production_t, r.yield_bags_ha, r.farmgate_price_ksh_bag, r.imports_t,
         "AFA Yearbook of Statistics 2025")
        for r in national_afa.itertuples(index=False)
    ],
)
conn.commit()

for table in ["counties", "county_production", "national_production"]:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table:<20} {count} rows")

counties             47 rows
county_production    235 rows
national_production  6 rows


In [4]:
def query(sql, params=()):
    """Run a SELECT and return the result as a DataFrame."""
    return pd.read_sql_query(sql, conn, params=params)

## 3. Queries

In [5]:
query("""
SELECT year,
       SUM(area_ha)                          AS total_area_ha,
       SUM(production_t)                     AS total_production_t,
       ROUND(SUM(production_t) / SUM(area_ha), 2) AS yield_t_ha
FROM county_production
GROUP BY year
ORDER BY year;
""")

,year,total_area_ha,total_production_t,yield_t_ha
0,2020,2171697.0,3795175.0,1.75
1,2021,2170503.0,3311941.0,1.53
2,2022,2113520.0,3082220.0,1.46
3,2023,2430013.0,4285206.0,1.76
4,2024,2407245.0,4028320.0,1.67


### Q2. Top 10 counties by average production (JOIN)

In [6]:
top10_sql = query("""
SELECT c.county_name,
       ROUND(AVG(p.production_t)) AS avg_production_t
FROM county_production AS p
JOIN counties AS c ON c.county_id = p.county_id
GROUP BY c.county_name
ORDER BY avg_production_t DESC
LIMIT 10;
""")
top10_sql

,county_name,avg_production_t
0,Uasin Gishu,432319.0
1,Trans Nzoia,421625.0
2,Bungoma,258707.0
3,Narok,258559.0
4,Nakuru,216010.0
5,Kakamega,197818.0
6,Nandi,131687.0
7,Migori,107680.0
8,Homa Bay,104766.0
9,Kericho,104588.0


### Q3. Yield ranking, excluding very small maize areas (WHERE + GROUP BY)

In [7]:
query("""
SELECT c.county_name,
       ROUND(SUM(p.production_t) / SUM(p.area_ha), 2) AS yield_t_ha
FROM county_production AS p
JOIN counties AS c ON c.county_id = p.county_id
WHERE c.small_area_flag = 0
GROUP BY c.county_name
ORDER BY yield_t_ha DESC;
""")

,county_name,yield_t_ha
0,Uasin Gishu,4.01
1,Trans Nzoia,3.78
2,Elgeyo-Marakwet,2.95
3,Bungoma,2.83
4,Nakuru,2.81
5,Kericho,2.69
6,Nandi,2.56
7,Kakamega,2.08
8,Narok,2.08
9,Nyandarua,1.93


### Q4. One county's record, using a parameter

In [8]:
query(
    """
    SELECT p.year, p.area_ha, p.production_t,
           ROUND(p.production_t / p.area_ha, 2) AS yield_t_ha
    FROM county_production AS p
    JOIN counties AS c ON c.county_id = p.county_id
    WHERE c.county_name = ?
    ORDER BY p.year;
    """,
    params=("Murang'a",),
)

,year,area_ha,production_t,yield_t_ha
0,2020,66381.0,63653.0,0.96
1,2021,62983.0,49256.0,0.78
2,2022,67530.0,47434.0,0.70
3,2023,68399.0,56827.0,0.83
4,2024,54189.0,55488.0,1.02


### Q5. Counties whose production fell from 2023 to 2024

In [8]:
query("""
SELECT c.county_name,
       p23.production_t AS production_2023_t,
       p24.production_t AS production_2024_t,
       ROUND((p24.production_t - p23.production_t) * 100.0 / p23.production_t, 1) AS change_pct
FROM county_production AS p23
JOIN county_production AS p24 ON p24.county_id = p23.county_id AND p24.year = 2024
JOIN counties AS c ON c.county_id = p23.county_id
WHERE p23.year = 2023
  AND p24.production_t < p23.production_t
ORDER BY change_pct;
""")

,county_name,production_2023_t,production_2024_t,change_pct
0,Kisii,119244.0,41087.0,-65.5
1,Kitui,36292.0,17845.0,-50.8
2,Kilifi,81813.0,41907.0,-48.8
3,Machakos,175493.0,91910.0,-47.6
4,Kisumu,105003.0,58657.0,-44.1
5,Samburu,15298.0,9135.0,-40.3
6,Nyandarua,46617.0,28698.0,-38.4
7,Embu,41327.0,25659.0,-37.9
8,Lamu,74318.0,48436.0,-34.8
9,Nyeri,35280.0,24832.0,-29.6


### Q6. County totals vs the national figure

In [10]:
query("""
SELECT n.year,
       SUM(p.production_t) AS county_sum_t,
       n.production_t      AS national_t,
       ROUND((SUM(p.production_t) - n.production_t) * 100.0 / n.production_t, 2) AS difference_pct
FROM county_production AS p
JOIN national_production AS n ON n.year = p.year
GROUP BY n.year
ORDER BY n.year;
""")

,year,county_sum_t,national_t,difference_pct
0,2020,3795175.0,3795170.0,0.00
1,2021,3311941.0,3314430.0,-0.08
2,2022,3082220.0,3089901.0,-0.25
3,2023,4285206.0,4285206.0,0.00
4,2024,4028320.0,4028320.0,0.00


## 4. Validate SQL against pandas

In [11]:
top10_pandas = (
    county_long.groupby("county")["production_t"].mean().round().sort_values(ascending=False).head(10)
)
match = top10_sql["county_name"].tolist() == top10_pandas.index.tolist()
print("SQL and pandas top-10 lists match:", match)

conn.close()

SQL and pandas top-10 lists match: True
